In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset
#from math_utils import acos, arcosh
#from product_distance_v2 import ProductDistance, acosh
import networkx as nx 
import math
import numpy as np
import matplotlib.pyplot as plt

from GraphRicciCurvature.OllivierRicci import OllivierRicci
from GraphRicciCurvature.FormanRicci import FormanRicci
    
import community.community_louvain as community_louvain

from sklearn import preprocessing, metrics

from torch_geometric.datasets import WebKB
from torch_geometric.utils import to_networkx, k_hop_subgraph
from torch_geometric.nn import GCNConv

import wandb

In [20]:
    run_ball_rewire = wandb.init(
        project = "Product-mfd-GAT",
        config = {
            "architecture": "Ball-bellman_rewire-radius=3-GAT-K=k",
            "dataset":"cornell",
            "epoch": 100,
            "lr": 0.001,
            "weight_decay":0.001,
            "Batch size": 1,
        }
    )

In [3]:
# if dataset.name == 'Cora':
#     data.train_mask[:1624] = True
#     data.train_mask[1624:2166] = True
#     data.train_mask[2166:] = True
# elif dataset.name == 'Citeseer':
#     data.train_mask[:1995] = True
#     data.train_mask[1995:2661] = True
#     data.train_mask[2661:] = True
# elif dataset.name == 'Pubmed':
#     data.train_mask[:11829] = True
#     data.train_mask[11829:15773] = True
#     data.train_mask[15773:] = True
# elif dataset.name == 'cornell':
#     data.train_mask[:110] = True
#     data.train_mask[110:147] = True
#     data.train_mask[147:] = True

In [21]:
device = torch.device("cuda")
dataset = WebKB(root="/home/siddy/META/data", name='cornell')
data = dataset[0].to(device)

In [22]:
feat = data.x
print(feat.shape)

torch.Size([183, 1703])


In [23]:
G = to_networkx(data, to_undirected=True)

In [24]:
def init_ball(radius: int, graph):
    edge_dict = {}
    for index, node in enumerate(graph.nodes()):
        paths = nx.single_source_shortest_path(graph, node, radius)
        if index not in edge_dict:
            edge_dict[index] = []
        for key, value in paths.items():
            if len(value) == 2:
                edge_dict[index].append(value)
            elif len(value)==3:
                edge_dict[index].append(value[1:])
    return edge_dict

In [34]:
edge_dict= init_ball(radius=3, graph=G)

In [35]:
print(edge_dict)

{0: [[0, 101], [0, 122], [101, 8], [101, 20], [101, 109]], 1: [[1, 27], [27, 74], [27, 97]], 2: [[2, 75], [2, 130], [75, 25], [75, 66]], 3: [], 4: [], 5: [], 6: [[6, 109], [6, 149], [109, 8], [109, 20], [109, 57], [109, 101], [109, 122], [149, 165]], 7: [], 8: [[8, 28], [8, 101], [8, 109], [8, 122], [8, 158], [28, 21], [28, 24], [28, 47], [28, 84], [101, 0], [101, 20], [109, 6], [109, 57], [158, 31], [158, 65], [158, 147]], 9: [[9, 148], [148, 57], [148, 67]], 10: [[10, 142], [10, 154], [10, 166], [154, 57]], 11: [], 12: [], 13: [[13, 150], [150, 66], [150, 89], [150, 110], [150, 146]], 14: [], 15: [], 16: [], 17: [], 18: [[18, 96], [96, 57], [96, 70], [96, 121], [96, 173]], 19: [[19, 103], [103, 57], [103, 83]], 20: [[20, 47], [20, 101], [20, 109], [20, 158], [47, 24], [47, 28], [101, 0], [101, 8], [101, 122], [109, 6], [109, 57], [158, 31], [158, 65], [158, 147]], 21: [[21, 28], [21, 41], [21, 147], [21, 180], [28, 8], [28, 24], [28, 47], [28, 84], [41, 66], [41, 67], [147, 57], [147

In [36]:
class Edge_atten(nn.Module):
    def __init__(self, in_channels, out_channels, num_heads=1, concat_heads=True, alpha=0.2):
        super().__init__()
        self.num_heads=num_heads
        self.concat_heads = concat_heads
        if self.concat_heads:
            assert out_channels % num_heads==0, "number of output channels must be multiple of count of heads"
            out_channels = out_channels // num_heads

        self.linear = nn.Linear(in_channels, out_channels*num_heads)
        self.a = nn.Parameter(torch.Tensor(num_heads, 2*out_channels))
        self.leakyrelu = nn.LeakyReLU(alpha)

        #xavier uniform initialization
        nn.init.xavier_uniform_(self.linear.weight.data, gain=1.414)
        nn.init.xavier_uniform_(self.a.data, gain=1.414)
    
    def forward(self, node_feats, edge_index):
        node_feats = torch.unsqueeze(node_feats, dim=0)
        batch_size, num_nodes = node_feats.size(0), node_feats.size(1)
        node_feats = self.linear(node_feats)
        node_feats = node_feats.view(batch_size, num_nodes, self.num_heads, -1)
        node_feats_flat = node_feats.view(batch_size*num_nodes, self.num_heads, -1)
        edge_indices_row = edge_index[0]
        edge_indices_col = edge_index[1]
        a_input = torch.cat([
            torch.index_select(input=node_feats_flat, index=edge_indices_row, dim=0),
            torch.index_select(input=node_feats_flat, index=edge_indices_col, dim=0)
        ], dim=-1)
        attn_logits = torch.einsum('bhc, hc->bh', a_input, self.a)
        attn_logits = self.leakyrelu(attn_logits)
        attn_probs = F.softmax(attn_logits, dim=-2)
        return attn_probs

In [37]:
    
def dot(x,y): return torch.sum(x * y, -1)
def acosh(x):
    return torch.log(x + torch.sqrt(x**2-1))
# def tanh(x, clamp=15):
#     return x.clamp(-clamp, clamp).tanh()
# def tan(x, clamp=15):
#     return x.clamp(-clamp, clamp).tan()
def tanh(x, clamp=15):
    return torch.clamp(torch.tanh(x), -clamp, clamp)

def tan(x, clamp=15):
    return torch.clamp(torch.tan(x), -clamp, clamp)

def acos(x): 
    return torch.acos(x)

class ProductDistance():
    def __init__(self, x, y, k):
        self.x = torch.tensor(x)
        self.y = torch.tensor(y)
        self.k = torch.tensor(k)
        if self.k < 0:
            self.k_ = torch.sqrt(-self.k)
        else:
            self.k_ = torch.sqrt(self.k)
        #self.eps = {torch.float32: 1e-7, torch.float64: 1e-15}
    
    @staticmethod
    def poincare_correct(x, eps=1e-10):
        current_norms = torch.norm(x,2,x.dim() - 1)
        mask_idx      = current_norms < 1./(1+eps)
        modified      = 1./((1+eps)*current_norms)
        modified[mask_idx] = 1.0
        #new_size      = [1]*current_norms.dim() + [x.size(x.dim()-1)]
        #return modified.unsqueeze(modified.dim()).repeat(*new_size)
        # return modified.unsqueeze(modified.dim()).expand(x.size())
        return modified.unsqueeze(-1)

    @staticmethod
    def poincare_proj(x, k, eps=1e-10):
        # if k<0:
        #     k = torch.sqrt(-k)
        # else:
        #     k = torch.sqrt(k)
        #x = x*ProductDistance.poincare_correct(x) correction should be after the projection 
        #print(f"poincare_correct:{x}")
        z = tanh(k*torch.norm(x, 2, -1))
        #print(f"p_exp_tanh:{z}")
        exp = torch.div(x*z, (k*torch.norm(x, 2, -1)))
        #print(f"p_exp:{exp}")
        #print(f"poincare_exp: {exp}")
        exp = exp*ProductDistance.poincare_correct(exp)
        return exp
        
    @staticmethod
    def hypersphere_proj(x, k):
        z = tan(k*torch.norm(x, 2, -1))
        #print(f"h_exp_tan:{z}")
        exp = torch.div(x*z, (k*torch.norm(x, 2, -1)))
        #print(f"h_exp: {exp}")
        return exp

    def hypersphere_dist(self, eps=1e-7):
        proj_x = ProductDistance.hypersphere_proj(self.x, self.k)
        proj_y = ProductDistance.hypersphere_proj(self.y, self.k)
        K = 1/self.k_
        #print(K)
        z  = 2*self.k*((torch.norm(proj_x-proj_y,2,-1))**2)
        #print(f"h_dist_z:{z}")
        uu =  1. + torch.div(z,((1+self.k*((torch.norm(proj_x,2,-1))**2))*(1+self.k*((torch.norm(proj_y,2,-1))**2))))
        #print(f"hypersphere_uu_for_clamp: {uu}")
        #return K*torch.acos(torch.clamp(uu), -1+eps, 1-eps
        valid_mask = (uu >= -1) & (uu <= 1)
        uu_valid = uu[valid_mask]
        uu_ = torch.empty_like(uu)
        uu_[valid_mask] = acos(uu_valid)  
        #h_ = acos(uu)
        #print(f"h_:{h_}")
        #clipped = torch.max(torch.min(x, max), min)
        min_, max_ = torch.tensor(-1+eps), torch.tensor(1-eps)
        h_dist = K*(torch.max(torch.min(uu_, max_), min_))
        #print(f"h_uu:{uu_}")
        #print(f"h_dist:{h_dist}")
        return h_dist

    def poincare_dist(self):
        proj_x = ProductDistance.poincare_proj(self.x, self.k)
        proj_y = ProductDistance.poincare_proj(self.y, self.k)
        K = 1/self.k_
        #print(K)
        z  = 2*self.k*((torch.norm(proj_x-proj_y,2,-1))**2)
        #print(f"p_dist_z:{z}")
        uu =  1. + torch.div(z,((1+self.k*(torch.norm(proj_x,2,-1)**2))*(1+self.k*(torch.norm(proj_y,2,-1)**2))))
        #print(f"p_uu:{uu}")
        #print(f"p_dist:{K*acosh(uu)}")
        return K*acosh(uu)

    def hypersphere_exp(x, k_):
        #self.x[..., 0] = torch.sqrt(1 - torch.norm(self.x[..., 1:],2,-1)**2)
        x[...,0] = k_
        n = torch.norm(x, 2, -1, keepdim=True)
        mask = torch.abs(n)<1e-7
        cos = torch.cos(n)
        cos[mask] = 1.0
        sin = torch.sin(n)
        sin[mask] = 0.0
        n[torch.abs(n)<1e-7] = 1.0
        e = cos*x + sin*x/n
        return e/torch.norm(e, 2, -1, True)


    def hypersphere_mfd(self, eps=1e-9): #to calculate the hypersphere distance on full mfd
        proj_x = ProductDistance.hypersphere_exp(self.x, self.k_)
        proj_y = ProductDistance.hypersphere_exp(self.y, self.k_)
        K = 1/self.k_
        dist_ = self.k_*torch.clamp(dot(proj_x, proj_y), -1+eps, 1-eps)
        return K*torch.acos(dist_)


    def euclidean_dist(self):
        """ Input shape (n, d) """
        #print(f"e_dist:{torch.norm(self.x-self.y, 2, dim=-1)}")
        return torch.norm(self.x-self.y, 2, dim=-1)


In [38]:
class BallGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.empty(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index, edge_weight:None):
        # what is the shape of inpur x ? - needed [N, in_channels]
        # edge indices shape needed is [2, E]

        #add self_loops to the adjacency matrix, how to give num nodes?
        #edge_index, _ = add_self_loops(edge_index)
        #print(edge_index)
        # linearly transform node feature matrix
        x = self.lin(x)
        #x = torch.index_select(input=x, index=edge_index[0], dim=0)
        # x_ball = torch.cat([torch.index_select(input=x, index=edge_index[0], dim=0), NOTE THAT IT WILL GIVE INDEX OUT OF RANGE ONE OPTION IS TO GO WITH REINDEXING
        #             torch.index_select(input=x, index=edge_index[1], dim=0)],dim=0)
        #compute normalization
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(0.5)
        deg_inv_sqrt[deg_inv_sqrt==float('inf')] = 0
        #print(deg_inv_sqrt.shape)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # propagating messages
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight, norm=norm)
        out = torch.index_select(input=out, index=min(edge_index[0]), dim=0) #NOTE TRICK IS TO PICK MIN EDGE INDEX AS IT WILL CORRESPOND TO THE CENTER NODE OF THE BALL
        # bias
        out += self.bias
        return torch.squeeze(out)

    def message(self, x_j, norm):
        # x_j has shape [E, out_channels]
        # normalize node features
        return norm.view(-1,1) *x_j

In [39]:
class BallGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = BallGCNConv(in_channels, hidden_channels)
        self.fc = Linear(hidden_channels, out_channels)
    def forward(self, x, edge_index, edge_weight):
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.fc(self.conv1(x, edge_index, edge_weight))
        return x

In [40]:
edge_attn = Edge_atten(in_channels=data.x.size(1), out_channels=64).to(device)
ball_gcn = BallGCN(in_channels=data.x.size(1), hidden_channels=64, out_channels=dataset.num_classes).to(device)
msg = nn.Linear(in_features=data.x.size(1), out_features=dataset.num_classes, bias=False).to(device)

In [41]:
optimizer = torch.optim.Adam(set(edge_attn.parameters())| set(ball_gcn.parameters()) | set(msg.parameters()), lr=0.001, weight_decay=0.001)


In [42]:
def ball_dist(edge_list):
    edge_list_nx = list(tuple(i) for x, i in enumerate(edge_list.t().numpy()))
    g_ball = nx.Graph()
    g_ball.add_edges_from(edge_list_nx)
    orc= OllivierRicci(g_ball, alpha=0.5, verbose="TRACE")
    orc.compute_ricci_curvature()
    G_orc = orc.G.copy()
    ricci_curvatures = nx.get_edge_attributes(G_orc, "ricciCurvature")
    #print(ricci_curvatures)
    dist_ball = {}
    for key, value in ricci_curvatures.items():
        #print(key)
        dist_ball[key] = []
        x, y = feat[key[0]], feat[key[1]]
        product_dist = ProductDistance(x,y,value)
        euclidean_dist = product_dist.euclidean_dist()
        poincare_dist = product_dist.poincare_dist()
        sphere_dist = product_dist.hypersphere_dist()
        dist_ = euclidean_dist+poincare_dist+sphere_dist
        dist_ = round(float(dist_.detach().cpu().numpy()),2)
        #print(dist_)
    # print(dist_ball[key])
        dist_ball[key].append(value)
        dist_ball[key].append(dist_)
    # print(dist_ball[key])
        # dist_ball[key].append(dist_)
    return dist_ball

    
def ball_rewiring(dist_ball):
    weighted_edge_list = []
    for edge, values in dist_ball.items():
        weighted_edge_list.append(edge+(values[1],))
    g_rewire = nx.Graph()
    g_rewire.add_weighted_edges_from(weighted_edge_list)
    sp_dict = {}
    for index, node in enumerate(g_rewire.nodes()):
        length, path = nx.single_source_bellman_ford(g_rewire, node, weight='weight')
        if index not in sp_dict:
             sp_dict[index] = []
        sp_dict[index].append(length)
        #sp_dict[index].append(path)
    return sp_dict

def rewired_edges(sp_dict):
    rewired_edges = []
    for index, neighbours in sp_dict.items():
        radius = []
        for node, dist in neighbours[0].items():
            radius.append(dist)
        #print(np.mean(radius))
        for node, dist in neighbours[0].items():
            if dist <= np.mean(radius):
                rewired_edges.append([index, node])
    return rewired_edges

def star_rewire(dist_ball):
  dist_ = []
  for edge, value in dist_ball.items():
    dist_.append(value[1])
  rewired_edges = []
  for edge, value in dist_ball.items():
    if value[1] <= np.mean(dist_):
      u,v = edge
      rewired_edges.append([u,v])
  return rewired_edges

In [43]:
def train():
    edge_attn.train()
    ball_gcn.train()
    msg.train()
    feats = []
    for node, ball in edge_dict.items():
        if len(ball)<1:
            index_feat = msg(data.x[node])
            #print(index_feat)
            #print(index_feat.shape)
            # np.vstack((feats, np.array(index_feat.detach())))
            feats.append(index_feat)
        else:
            edge_list = torch.permute(torch.tensor(ball, dtype=torch.long), (1,0))
            product_distances = ball_dist(edge_list)
            shortest_paths = ball_rewiring(product_distances)
            edge_list = rewired_edges(shortest_paths)
            #edge_list = star_rewire(product_distances)
            #print(rewired_edges)
            if len(edge_list)<1:
              index_feat = msg(data.x[node])
              feats.append(index_feat)
            else:
              edge_list = torch.permute(torch.tensor(edge_list, dtype=torch.long), (1,0)).to(device)
              attn_probs = edge_attn(data.x, edge_list)
              # here ball should get the distances and recompute the ball
              out = ball_gcn(data.x, edge_list, attn_probs)
              # print(out)
              # print(out.shape)
              # np.vstack((feats, np.array(out.detach())))
              feats.append(out)
    feat = torch.stack(feats, -2)
    #print(feat)
    print(feat.shape)
    loss = F.cross_entropy(feat[data.train_mask[:,0]], data.y[data.train_mask[:,0]])
    loss.backward()
    optimizer.step()
    return float(loss)

In [44]:
@torch.no_grad()
def test():
    edge_attn.eval()
    ball_gcn.eval()
    msg.eval()
    feats = []
    for node, ball in edge_dict.items():
        if len(ball)<1:
            index_feat = msg(data.x[node]).argmax(dim=-1)
            #print(index_feat)
            #print(index_feat.shape)
            # np.vstack((feats, np.array(index_feat.detach())))
            feats.append(index_feat)
        else:
            edge_list = torch.permute(torch.tensor(ball, dtype=torch.long), (1,0))
            product_distances = ball_dist(edge_list)
            shortest_paths = ball_rewiring(product_distances)
            edge_list = rewired_edges(shortest_paths)
            #edge_list = star_rewire(product_distances)
            if len(edge_list)<1:
              index_feat = msg(data.x[node]).argmax(dim=-1)
              feats.append(index_feat)
            else:
              edge_list = torch.permute(torch.tensor(edge_list, dtype=torch.long), (1,0)).to(device)
              attn_probs = edge_attn(data.x, edge_list)
              # here ball should get the distances and recompute the ball
              out = ball_gcn(data.x, edge_list, attn_probs).argmax(dim=-1)
              # print(out)
              # print(out.shape)
              # np.vstack((feats, np.array(out.detach())))
              feats.append(out)
    feat = torch.stack(feats, -1)
    #print(feat)
    #print(feat.shape)
    accs=[]
    for mask in [data.train_mask[:,0], data.val_mask[:,0], data.test_mask[:,0]]:
        accs.append(int((feat[mask] == data.y[mask]).sum()) / int(mask.sum()))
    return accs 

In [45]:
import time
epochs=100
best_val_acc = final_test_acc = 0
times = []
for epoch in range(1, epochs+1):
    start = time.time()
    loss= train()
    wandb.log({"Loss":loss})
    train_acc, val_acc, tmp_test_acc = test()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        test_acc = tmp_test_acc
    print(f"Epoch:{epoch}, Loss:{loss}, Train:{train_acc}, Val:{val_acc}, Test:{test_acc}")
    wandb.log({"train_acc":train_acc})
    wandb.log({"test_acc":test_acc})
    wandb.log({"val_acc":val_acc})
    times.append(time.time()-start)
print(f"Median time per epoch: {torch.tensor(times).median():.4f}s")

/tmp/ipykernel_9204/1918337355.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.x = torch.tensor(x)
/tmp/ipykernel_9204/1918337355.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y)


torch.Size([183, 5])
Epoch:1, Loss:1.6145142316818237, Train:0.26436781609195403, Val:0.1864406779661017, Test:0.13513513513513514
torch.Size([183, 5])
Epoch:2, Loss:1.5859229564666748, Train:0.3333333333333333, Val:0.1864406779661017, Test:0.13513513513513514
torch.Size([183, 5])


KeyboardInterrupt: 

In [46]:
run_ball_rewire.finish()

Loss,█▅▁
test_acc,▁▁
train_acc,▁█
val_acc,▁▁
Loss,1.55252
test_acc,0.13514
train_acc,0.33333
val_acc,0.18644


- store the curvature of graph into a list may be -- curvature is edge wise
- feed curvature, x, y, edge atten into product manifold class and get distances for each node pairs
- make a nx graph having these distances as edge weight and then do message passing on re-wiried graph

In [ ]:
edge_list = torch.permute(torch.tensor(edge_dict[0], dtype=torch.long), (1,0))

In [ ]:
edge_list_nx = list(tuple(i) for x,i in enumerate(edge_list.t().numpy()))

In [ ]:
edge_list_nx

In [ ]:
g_ball = nx.Graph()
g_ball.add_edges_from(edge_list_nx)

In [ ]:
orc = OllivierRicci(g_ball, alpha=0.5, verbose="TRACE")

In [ ]:
orc.compute_ricci_curvature()
G_orc = orc.G.copy() 

In [ ]:
ricci_curvtures = nx.get_edge_attributes(G_orc, "ricciCurvature")

In [ ]:
ricci_curvtures # amazingly it gives dict of edge: curvatures

In [ ]:
def show_results(G, curvature="ricciCurvature"):

    # Print the first five results
    print("Ball graph of center node 0")
    for n1,n2 in list(G.edges())[:5]:
        print("Ricci curvature of edge (%s,%s) is %f" % (n1 ,n2, G[n1][n2][curvature]))

    # Plot the histogram of Ricci curvatures
    plt.subplot(2, 1, 1)
    ricci_curvtures = nx.get_edge_attributes(G, curvature).values()
    plt.hist(ricci_curvtures,bins=20)
    plt.xlabel('Ricci curvature')
    plt.title("Histogram of Ricci Curvatures G_ball")

    # Plot the histogram of edge weights
    plt.subplot(2, 1, 2)
    weights = nx.get_edge_attributes(G, "weight").values()
    plt.hist(weights,bins=20)
    plt.xlabel('Edge weight')
    plt.title("Histogram of Edge weights G_ball")

    plt.tight_layout()

show_results(G_orc)

In [ ]:
edge_atten = Edge_atten(in_channels=data.x.size(1), out_channels=64)

In [ ]:
atten_probs = edge_atten(data.x, edge_list)
weights = torch.squeeze(atten_probs).detach().numpy()

In [ ]:
weights

In [ ]:
for weight in weights:
    print(weight)

In [ ]:
ball_dict = {}
for i,(key, value) in enumerate(ricci_curvtures.items()):
    key, value = (key, value)
    ball_dict[key] = []
    ball_dict[key].append(value)
    ball_dict[key].append(weights[i])

In [ ]:
ball_dict

In [ ]:
key_list = list(ball_dict.keys())

In [ ]:
key_list[0]

In [ ]:
val_list = list(ball_dict.values())

In [ ]:
val_list[0][0]

In [ ]:
x, y = key_list[0]
x, y = torch.tensor(x), torch.tensor(y)
k = val_list[0][0]
atten_weight = val_list[0][1]


In [ ]:
x, y = feat[x], feat[y]

In [ ]:
print(k)

In [ ]:
print(atten_weight)

In [ ]:
K = (1. / torch.sqrt(torch.tensor(-k)))
print(K)

In [ ]:
dist = (acosh(torch.tensor(k)*atten_weight))
print(dist)

In [ ]:
(torch.acos(torch.tensor(k)*atten_weight))

In [ ]:
z = 2*torch.tensor(k)*torch.norm(x-y,2,0)**2
print(z)

In [ ]:
dist = 1. - torch.div(z, ((1+torch.tensor(k)*torch.norm(x,2,0)**2)*(1+torch.tensor(k)*torch.norm(y,2,0)**2)))
print(dist)

In [ ]:
def acosh(x):
    return torch.log(x + torch.sqrt(x**2-1))

In [ ]:
x, y = feat[x], feat[y]
print(x)
print(y)

In [ ]:
product_distance = ProductDistance(x, y, k, atten_weight)

In [ ]:
print("Product Distance:", product_distance.productdistance())
print("hyperesphere Distance:", product_distance.dist_s())
print("poincare Distance:", product_distance.dist_p())
print("euclideea Distance:", product_distance.dist_e())